# 04 · Error analysis and export

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egumasa/lda2-final-template/blob/main/notebooks/04_report.ipynb)

Show an item the model got wrong, and say whose fault it was.

```
  01_build_pool_<track>  →  02_sample  →  03_annotate  →▶ 04_prompt  →  05_report
```

| | |
|---|---|
| **Reads** | the gold set (02) · the frozen predictions and rounds (03) |
| **Writes** | `outputs/` — the predictions CSV, the report scaffold, a copy of your gold set |

---

This is the highest-value part of the whole project, and the one the Q&A will definitely go to. A low F1 with a clear account of *why* is worth more than a high one without.

## Setup — run this first

This cell mounts your Google Drive and finds your group's shared folder, `lda2-final-template`. Everything the project produces — the pool, the gold set, your prompts, the outputs — is an ordinary file in there, which is what makes it survive the runtime resetting *and* lets the rest of your group see it.

**One member sets the folder up once:**

1. That member runs the `git clone` line this cell prints if the folder is missing, which puts it in their own Drive.
2. They share it with the group (right-click ▸ *Share*), with edit access.
3. Everyone else opens *Shared with me*, right-clicks the folder, and chooses **Add shortcut to Drive** ▸ *My Drive*.

Keep that shortcut's name exactly `lda2-final-template`. It is what makes the same path work for all of you — if Drive renames it to `lda2-final-template (1)`, this cell will not find it.

From then on, open notebooks from the folder itself (*File ▸ Open notebook ▸ Drive*) rather than from the GitHub badge, so you are working on your group's copy and not a fresh one.

In [ ]:
# ------------------------------------------------------------------
# SETUP — run me first. You are not expected to read it.
# ------------------------------------------------------------------
# This cell is plumbing, and it is the only cell in the project that is.
# It finds your group's shared folder in Google Drive, because everything
# this project keeps goes in there: a Colab runtime is wiped when it resets,
# and nobody else in your group can see inside it. Then it makes the
# project's own code importable. Run it and move on; nothing below asks you
# to have understood it.

FOLDER = "lda2-final-template"     # the shared folder, in every member's Drive

import os, sys

PROJECT = ".."                              # running locally: it is just above us

try:
    from google.colab import drive           # only exists inside Colab
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/" + FOLDER
    if not os.path.isdir(PROJECT):
        raise RuntimeError(
            "Could not find " + PROJECT + "\n\n"
            "Setting the folder up for your group? Run this in a new cell:\n"
            "  !git clone https://github.com/egumasa/lda2-final-template.git "
            + PROJECT + "\n"
            "then share the folder with the rest of your group.\n\n"
            "Someone else already did? Open Drive, find the folder under "
            "'Shared with me', right-click it, and choose 'Add shortcut to "
            "Drive'. Keep the name exactly " + FOLDER + ".")
    # Work inside the project folder, where the notebooks live.
    os.makedirs(PROJECT + "/notebooks", exist_ok=True)
    os.chdir(PROJECT + "/notebooks")

# scripts/ and config.py, by their real paths - so they are found from wherever
# this notebook happens to be working.
sys.path.append(PROJECT)
sys.path.append(PROJECT + "/scripts")

# Re-read config.yaml every time this cell runs. Without the reload, Python
# hands back the settings it read the FIRST time, and editing config.yaml
# would appear to do nothing until you restarted the runtime.
import importlib
import config
importlib.reload(config)

# Named one by one rather than with `import *`, so that every name a cell
# below uses can be traced back to the file it came from — config.yaml for
# these, scripts/ for the rest.
from config import (TRACK, GROUP, RUN, SEED, N_PER_CLASS, MEMBERS,
                    LABELS_ORDER, ROOT, OUT_DIR, POOL_PATH, DEMO_POOL_PATH,
                    SAMPLE_PATH, GOLD_PATH, PRED_PATH, ROUNDS_PATH,
                    PROMPT_FILE, SHEET_PATH, TRIAGE_PATH, describe)

# Loading files is plumbing. The scoring, the error table and the triage are
# the analysis this notebook is about, so they are in the notebook, below.
from pipeline import (load_gold, load_predictions, load_json, save_json,
                      export_results)
from annotate import remembered_sheet, load_coder_sheets, disagreements

describe()                  # what this notebook is working on


> **Everything above comes from `config.yaml`** — one small file at the top of the repo, which you edit once as a group, and the only file in the plumbing you touch. That is deliberate: the seed that drew your sample has to be the seed you report, and five copies of a number in five notebooks is five chances for them to disagree. Your settings are also the filenames — `track: cars50`, `group: kimura`, `run: v1` means this notebook reads and writes `cars50_kimura_v1_...`. If the line it just printed is not your track, your group and your seed, fix `config.yaml` and re-run this cell.

In [ ]:
# ══ STEP 1 · Load the frozen run ══════════════════════════════════════════
# Goal      : the gold set, the predictions file, and the per-round table.
# Available : load_gold(GOLD_PATH)  ->  gold
#             load_predictions(PRED_PATH)  ->  pred_final
#             load_json(ROUNDS_PATH, what="rounds")  ->  f1_by_round
# Source    : scripts/pipeline.py · load_gold, load_predictions, load_json
# Pointer   : Day 2 S6 — loading a frozen predictions file is exactly what you did there.
# Produce   : gold · pred_final · f1_by_round      ← later cells use these names
# Note      : nothing in this notebook calls the model. If a number here
#             differs from notebook 04, you are loading a different file —
#             not watching the model change its mind.

# ✏️ your code here — fill in each ____

gold = load_gold(____)                 # GOLD_PATH
pred_final = load_predictions(____)    # PRED_PATH
f1_by_round = load_json(____, what="rounds")      # ROUNDS_PATH

print(len(gold), "gold ·", len(pred_final), "predictions")
f1_by_round


### The code that does it — read it, then run it

The functions the rest of this notebook is made of. `evaluate` and `show_errors` are the same ones you ran in 03. `errors_on_disagreed` and `triage_counts` are new, and they are what turns a list of mistakes into an argument.

It is read straight out of `scripts/` when this notebook is generated, so it is not a simplified copy: it is the code that runs. Two things to look for as you read:

- **`show_errors` keeps only the rows where gold and prediction differ**, and returns them as a `DataFrame` — a table, which is why Colab draws it nicely and why you can filter it with `errors[errors.gold == "…"]` below.
- **`errors_on_disagreed` does one join**, on the item id. Look at how little there is to it — the whole force of that number comes from the fact that you built both tables yourselves, from the same forty items.
- **Nothing here calls the model.** Every function takes lists you already loaded. That is what "frozen" means: from here on your numbers can only change if you load a different file.

Run the cell to define these, then use them in the step below.

In [ ]:
# Read straight out of scripts/metrics.py — this IS the code that runs.

import pandas as pd
from sklearn.metrics import (classification_report, confusion_matrix,
                             cohen_kappa_score, f1_score)
from pipeline import (plot_confusion_matrix, label_set,
                      triage_category, TRIAGE_CATEGORIES)

def evaluate(gold, predictions, ordered=False, labels=None, title="Confusion matrix"):
    """Score predictions against gold: per-class P/R/F1 + macro, Cohen's kappa, and a
    confusion-matrix heatmap. Returns the macro-F1 as a number.

    ordered=True adds QUADRATIC WEIGHTED kappa — use it only when the labels sit on a
    scale (A1 < A2 < ... < C2), so that a near miss counts as a smaller error than a
    far one. For unordered categories, plain kappa is the one to report.

    IMPORTANT for ordered=True: the scale is taken from `labels`, in the order given.
    Left off, `labels` is read off the gold set and sorted ALPHABETICALLY — which is
    correct for A1..C2 and Move 1..3, but wrong for something like Low/Mid/High
    (alphabetical puts High first). If your labels are ordered and not alphabetical,
    pass them yourself: evaluate(gold, pred, ordered=True, labels=LABELS_ORDER).
    """
    # --- Compatibility with the older 4-positional call form -----------------------
    # An earlier version of this file took evaluate(gold, predictions, labels, title).
    # If we were called that way, argument 3 is a list of labels rather than a
    # true/false flag. Rather than fail with a confusing error - or worse, silently
    # treat a non-empty list as "ordered=True" - detect it and shuffle the arguments.
    if isinstance(ordered, (list, tuple)):
        print("NOTE: old call form evaluate(gold, pred, labels, title) — treating "
              "argument 3 as labels. The current form is "
              "evaluate(gold, pred, ordered=..., labels=...).")
        if isinstance(labels, str):
            title = labels
        labels = list(ordered)
        ordered = False

    ### Step 1: line the two label lists up, gold first ###
    y_true = []                          # the correct labels, from the gold set
    for item in gold:
        y_true.append(item["label"])
    y_pred = predictions                 # the model's labels, in the same order

    if labels is None:
        labels = label_set(gold)

    ### Step 2: per-class precision / recall / F1, as a text table ###
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

    ### Step 3: one overall number — agreement corrected for chance ###
    # Only meaningful if there is more than one label to be right or wrong about.
    if len(set(labels)) < 2:
        print("Cohen's kappa            undefined (only one label present)")
    else:
        print(f"Cohen's kappa            {cohen_kappa_score(y_true, y_pred):.3f}")
        if ordered:                      # only when the labels sit on a scale
            weighted = cohen_kappa_score(y_true, y_pred, labels=labels,
                                         weights="quadratic")   # near misses hurt less
            print(f"Cohen's kappa (weighted) {weighted:.3f}   <- labels are ordered")
            # Say WHICH order we used, so a wrong one is visible rather than silent.
            print("  scale order used:", " < ".join(labels))

    ### Step 4: draw the same information as a picture ###
    matrix = confusion_matrix(y_true, y_pred, labels=labels)
    plot_confusion_matrix(matrix, labels, title)

    ### Step 5: one number to carry from round to round ###
    macro_f1 = f1_score(y_true, y_pred, labels=labels,
                        average="macro", zero_division=0)
    return macro_f1

def show_errors(gold, predictions):
    """The items the model got wrong, as a table you can read and argue about."""
    rows = []
    for item, predicted in zip(gold, predictions):
        if item["label"] != predicted:
            row = {
                "id": item["id"],
                "gold": item["label"],
                "pred": predicted,
                "text": item["text"],
            }
            rows.append(row)
    print(f"{len(rows)} of {len(gold)} wrong.")
    return pd.DataFrame(rows)             # a table, so Colab displays it nicely

def errors_on_disagreed(errors, disagreed):
    """How many of the model's errors land on items YOUR OWN coders disagreed about.

    This is the most interesting number in the project. If the model's misses cluster
    on the items CoderA and CoderB could not agree on either, then what you have
    measured is a fuzzy boundary in the annotation scheme, not a stupid model - and
    that is a better finding than a clean F1.

    Both tables carry the same ids: the sheet was built from the sampled items, and
    the gold set was rebuilt from the sheet, so nothing has been renumbered in between.
    The sheet returns its ID column as text, though, so it is converted here.
    """
    if errors is None or disagreed is None:
        print("Nothing to compare - one of the two tables is empty.")
        return []

    error_ids = []
    for value in errors["id"]:
        error_ids.append(int(value))

    disagreed_ids = []
    for value in disagreed["ID"]:
        try:
            disagreed_ids.append(int(value))
        except (TypeError, ValueError):
            # A typed-over ID cell. to_canonical reports these too; skip it here
            # rather than lose the whole comparison over one bad row.
            pass

    overlap = []
    for item_id in error_ids:
        if item_id in disagreed_ids and item_id not in overlap:
            overlap.append(item_id)
    overlap.sort()

    if len(error_ids) == 0:
        print("No errors to compare.")
        return []

    share = len(overlap) / len(error_ids)
    print(len(error_ids), "errors.", len(overlap), "of them",
          "(" + format(share, ".0%") + ")",
          "are on items your two coders also disagreed about.")
    if overlap:
        print("  those ids:", overlap)
        print("  Read those items again before you blame the model.")
    return overlap

def triage_counts(triage, errors=None):
    """Count a triage by category, and say what it adds up to.

    `triage` is the dict your group writes by hand: {item id: "category - reason"}.
    The categories are fixed (model / scheme / wording / ambiguous) so that the counts
    mean the same thing across groups, and so this is a judgment you make from a menu
    rather than an essay you write.

    Pass `errors` (the table from show_errors) and it will also tell you how much of
    the error set you have actually been through.
    """
    counts = {}
    for category in TRIAGE_CATEGORIES:
        counts[category] = 0

    unrecognised = []
    for item_id in triage:
        category = triage_category(triage[item_id])
        if category is None:
            unrecognised.append(item_id)
        else:
            counts[category] = counts[category] + 1

    parts = []
    for category in TRIAGE_CATEGORIES:
        if counts[category] > 0:
            parts.append(str(counts[category]) + " " + category)
    if parts:
        print("Triaged " + str(len(triage)) + " errors: " + " / ".join(parts))
    else:
        print("Triaged 0 errors.")

    if unrecognised:
        print("NOTE: these lines do not start with one of",
              ", ".join(TRIAGE_CATEGORIES) + ":", unrecognised)
        print("      Start each line with the category word, then the reason:")
        print('        7: "scheme - Move 1/Move 2 boundary, our coders split too"')

    # "We looked at 3 of 40 errors" and "we looked at all 12" are different claims, and
    # the report should not let the first quietly read as the second.
    if errors is not None and hasattr(errors, "shape"):
        total = errors.shape[0]
        if len(triage) < total:
            print("You have triaged", len(triage), "of", total, "errors. Say so in the",
                  "report, or work through the rest.")
    return counts

In [ ]:
# ══ STEP 2 · Score the frozen run ═════════════════════════════════════════
# Goal      : the headline numbers, from the file rather than from a live run.
# Available : evaluate(gold, pred_final, ordered=..., labels=LABELS_ORDER)  ->  macro-F1
# Source    : the cell just above · scripts/metrics.py · evaluate
# Pointer   : Day 2 S6 Part B · Day 3 — the identical call.
# Produce   : macro_f1 — the numbers for report section 3      ← later cells use these names
# Note      : evaluate prints per-class P/R/F1 and kappa as well as the
#             macro average. "Which class is it worst at" is a more useful
#             sentence than "F1 = .62" — read the table, not just the
#             headline.
# Careful   : `annotator_agreement()` in notebook 03 compares two
#             ANNOTATORS. This compares gold against a MODEL. Same kind of
#             number, different claim — do not swap them in the report.

# ✏️ your code here — fill in each ____

macro_f1 = evaluate(gold, pred_final,
                    ordered=____,        # True only if your labels are a SCALE
                    labels=LABELS_ORDER)
print("macro-F1:", round(macro_f1, 3))


---

# Step 3 — Error analysis

**This is the part of the project the Q&A will actually go to.** A low F1 with a clear account of *why* beats a high one without, every time — and the account is only available to you because you built the gold set yourselves.

`show_errors` gives you every item the model got wrong. Very different findings live in that one table, and saying which is which is the whole job:

| | |
|---|---|
| **`model`** | the label is clear, both your coders agreed at once, and the model still missed it |
| **`scheme`** | the item is genuinely borderline *under your scheme* — and you know which ones these are, because you argued about them |
| **`wording`** | the label *name* misleads. `Gap` may read to a model as "missing data". This one your next prompt could fix |
| **`ambiguous`** | the item itself is unclear in a way no scheme would settle |

The difference between `scheme` and `wording` is worth being careful about: one of them is fixable by prompting and the other is not, and confusing them is how groups spend three rounds on a problem no prompt can reach.

In [ ]:
# ══ STEP 3 · The errors, and where they land ══════════════════════════════
# Goal      : get the misses, and see how many fall on items your own coders split on.
# Available : show_errors(gold, pred_final)  ->  a table of the items it got wrong
#             errors_on_disagreed(errors, disagreed)  ->  the overlapping ids
#             load_coder_sheets(SHEET_ID, CODERS) · disagreements(...)   (as in 03)
# Source    : the cell above · scripts/metrics.py · show_errors, errors_on_disagreed
# Pointer   : Day 3, "where is your best prompt still wrong?" — identical call.
# Produce   : errors · disagreed      ← later cells use these names
# Note      : the overlap number is the strongest claim this project can
#             support — that the model fails where your SCHEME is fuzzy,
#             rather than at random. Write it down either way: a LOW
#             overlap is just as reportable, and means something else.
# Careful   : re-reading the sheet needs the same coder names you used in
#             notebook 03. Skip these lines if the sheet is gone — the
#             triage below still works without it.

# ✏️ your code here — fill in each ____

errors = show_errors(gold, pred_final)
errors.head(15)

# `errors` is a table. To look at one label's misses, keep the rows whose
# gold column equals it — put in one of YOUR labels, spelled as LABELS has
# been printing them:
errors[errors.gold == "____"]

# Now the cross-reference: your coders' disagreements, back from the sheet.
CODERS = ["CoderA", "CoderB"]        # the same names you used in 03
SHEET_ID = remembered_sheet(SHEET_PATH)
rows = load_coder_sheets(SHEET_ID, CODERS)
disagreed = disagreements(rows, coders=CODERS)

overlap = errors_on_disagreed(errors, disagreed)


## Step 4 — Triage: say what each error is *caused by*

Now the judgment. Go through the errors — **all of them if there are few, at least eight or ten if there are many** — and write down, for each, which of the four things it is and how you know. Do this **together, out loud**, reading the actual sentences. It is the last genuinely analytical thing in the project and it takes about fifteen minutes.

Start each line with the category word, then a reason:

```python
TRIAGE = {
     7: "scheme  — Move 1/Move 2 boundary; our coders split on this one too",
    12: "model   — 'The aim of our study was' is about as clear as Move 3 gets",
    23: "wording — the model reads any citation as Move 1",
}
```

The ids come from the `errors` table above, and the ones in `overlap` are the obvious candidates for `scheme` — you have independent evidence for those. **A reason, not a verdict**: *"model — wrong"* is not worth writing down.

Two things come out of this. The counts go into report §4, so *"of 14 errors, 6 are our scheme's"* is a finding rather than an impression. And every `wording` line is a concrete next prompt round, which is what to say when someone asks what you would do with another week.

In [ ]:
# ══ STEP 4 · Triage the errors ════════════════════════════════════════════
# Goal      : attribute each miss, from the four categories, with a reason.
# Available : TRIAGE_CATEGORIES  ->  model · scheme · wording · ambiguous
#             triage_counts(triage, errors)  ->  the counts, and what is left to do
#             save_json(TRIAGE, TRIAGE_PATH, what="triaged errors")
# Source    : the cell above · scripts/metrics.py · triage_counts
# Pointer   : new — but it is the same judgment you made adjudicating in notebook 03.
# Produce   : TRIAGE      ← later cells use these names
# Note      : write it while the errors are in front of you. This is the
#             single hardest thing to reconstruct a week later.
# Ask       : do your `scheme` ids overlap with `overlap` from step 3? If
#             they do, say so — your own coders are the evidence.

# ✏️ your code here — fill in each ____

TRIAGE = {
    ____: "____ — ____",       # id: category — why you say so
    ____: "____ — ____",
    ____: "____ — ____",
}

triage_counts(TRIAGE, errors)
save_json(TRIAGE, TRIAGE_PATH, what="triaged errors")


## Step 5 — Export

Writes your gold set, a per-item predictions CSV, and a one-page report scaffold with the five required sections, all stamped with your group name.

The scaffold fills in what it can compute — labels, counts, the F1-per-round table, and now your triage: the category counts, and your reason printed beside each item. The *italic* placeholders are what is left for you: the QC narrative, the pattern in your `scheme` errors, and limitations that apply to **your** run rather than the generic three. A section left as the placeholder scores zero, so this is the start of the writing, not the end.

In [ ]:
# ══ STEP 5 · Export ═══════════════════════════════════════════════════════
# Goal      : write the gold set, the predictions CSV, and the report scaffold.
# Available : export_results(TRACK, gold, pred_final, f1_by_round, OUT_DIR, group=GROUP,
#                            run=RUN, triage=TRIAGE)
# Source    : scripts/pipeline.py · export_results
# Pointer   : new — but it only writes down what you already have.
# Produce   : three files in ../outputs/      ← later cells use these names
# Note      : pass triage=TRIAGE and section 4 becomes your analysis. Leave
#             it off and section 4 is a placeholder asking you for it.

# ✏️ your code here — fill in each ____

export_results(TRACK, gold, pred_final, f1_by_round, OUT_DIR,
               group=GROUP, run=RUN, triage=TRIAGE)


---

## Hand it in

One command collects everything into a folder next to the repo, keeping the `scripts/ · prompts/ · data/ · notebooks/ · outputs/` layout — because that layout *is* the reproducibility checklist from S10, and because the notebooks' paths only resolve if it stays intact.

```bash
python scripts/make_submission.py --group groupA
```

It deliberately leaves out `.git/`, `.venv/`, your `.env` (**it holds your API key**), the big pools in `data/pools/`, and anything ICNALE-derived.

Then: find the folder in Drive → right-click → **Download** → upload the zip to the *Final mini-project* assignment in Google Classroom → **Turn in**. One submission per group, with every member's name in `PLAN.md`.

Before you do, check that **all five notebooks run top to bottom on a fresh runtime**, in order. If they only work in the session where you built them piece by piece, they do not yet reproduce — and 02 through 05 handing files to each other is exactly what makes that checkable.

In [ ]:
# Optional: build the bundle from here instead of a terminal.
# !cd .. && python scripts/make_submission.py --group $GROUP
